In [ ]:
import sys

try:
    import numpy as np
    import matplotlib.pyplot as plt
    import csv
except ModuleNotFoundError:
    !{sys.executable} -m pip install numpy matplotlib scipy
    import numpy as np
    import matplotlib.pyplot as plt
    import csv

# Donnée à compléter
H = 0.20  # en m

# Lecture CSV
t, z = [], []
with open('mesures.csv', 'r', newline='') as Source:
    Lecteur = csv.reader(Source, delimiter=";")
    for row in Lecteur:
        t.append(float(row[0]))
        z.append(float(row[1]))

# Conversion en tableaux NumPy (indispensable)
t = np.array(t, dtype=float)
z = np.array(z, dtype=float)

# Découpage en 2 régimes
mask1 = z > H      # régime 1 : z > H
mask2 = z < H      # régime 2 : z <= H

t1, z1 = t[mask1], z[mask1]
t2, z2 = t[mask2], z[mask2]

if len(t1) < 2 or len(t2) < 3:
    raise ValueError("Pas assez de points pour ajuster les modèles (linéaire: 2 pts, parabole: 3 pts).")

# Ajustements

# Régime 1 : modèle linéaire z(t) = a1 + b1 t
b1, a1 = np.polyfit(t1, z1, 1)
z1_pred = a1 + b1 * t1
r2_1 = 1 - np.sum((z1 - z1_pred) ** 2) / np.sum((z1 - np.mean(z1)) ** 2)

# Régime 2 : modèle parabolique z(t) = a2 t^2 + b2 t + c2
a2, b2, c2 = np.polyfit(t2, z2, 2)
z2_pred = a2 * t2**2 + b2 * t2 + c2
r2_2 = 1 - np.sum((z2 - z2_pred) ** 2) / np.sum((z2 - np.mean(z2)) ** 2)

# Tracés : deux fenêtres
plt.close("all")

# Figure 1 : régime z > H
plt.figure()
plt.scatter(t1, z1, marker='+', label="Données $z > H$")
tt1 = np.linspace(t1.min(), t1.max(), 300)
plt.plot(tt1, a1 + b1 * tt1, color = "red", label="Modèle linéaire")

plt.xlabel("Temps $t\\ (\\mathrm{s})$")
plt.ylabel("Altitude $z\\ (\\mathrm{m})$")
plt.title("Régime 1 : $z > H$")
plt.grid(True)

eq1 = (
    rf"$z(t) = {a1:.3g} {b1:+.3g}\,t$" "\n"
    rf"$R^2 = {r2_1:.4f}$"
)
plt.text(0.05, 0.05, eq1,transform=plt.gca().transAxes,va="bottom")
plt.legend()

# Figure 2 : régime z < H 
plt.figure()
plt.scatter(t2, z2, marker='+', label="Données $z < H$")
tt2 = np.linspace(t2.min(), t2.max(), 500)
plt.plot(tt2, a2 * tt2**2 + b2 * tt2 + c2, color = "red", label="Modèle parabolique")

plt.xlabel("Temps $t\\ (\\mathrm{s})$")
plt.ylabel("Altitude $z\\ (\\mathrm{m})$")
plt.title("Régime 2 : $z < H$ (modèle parabolique)")
plt.grid(True)

eq2 = (
    rf"$z(t) = {a2:.3g}\,t^2 {b2:+.3g}\,t {c2:+.3g}$"
    "\n"
    rf"$R^2 = {r2_2:.4f}$"
)
plt.text(0.05, 0.05, eq2,transform=plt.gca().transAxes,va="bottom")
plt.legend()

plt.show()